# Use AutoAI and timeseries data with supporting features to predict PM2.5 by `ibm-watsonx-ai`

This notebook contains the steps and code to demonstrate support of AutoAI experiments for timeseries data sets in watsonx.ai Runtime service. It introduces commands for data retrieval, training experiments, persisting pipelines, testing pipelines, deploying pipelines, and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goals

The learning goals of this notebook are:

-  Define watsonx.ai Runtime experiment for timeseries data sets with supporting features. 
-  Work with experiments to train AutoAI models.
-  Compare trained models quality and select the best one for further deployment.
-  Online deployment and score the trained model.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#1.-Set-up-the-environment)
2. [Timeseries data set](#2.-Timeseries-data-set)
3. [Optimizer definition](#3.-Optimizer-definition)
4. [Experiment run](#4.-Experiment-run)
5. [Pipelines comparison](#5.-Pipelines-comparison)
6. [Deploy and Score](#6.-Deploy-and-Score)
7. [Cleanup](#7.-Cleanup)
8. [Summary and next steps](#8.-Summary-and-next-steps)

<a id="1.-Set-up-the-environment"></a>
## 1. Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).
-  Create a <a href="https://console.bluemix.net/catalog/infrastructure/cloud-object-storage" target="_blank" rel="noopener no referrer">Cloud Object Storage (COS)</a> instance (a lite plan is offered and information about how to order storage can be found <a href="https://console.bluemix.net/docs/services/cloud-object-storage/basics/order-storage.html#order-storage" target="_blank" rel="noopener no referrer">here</a>). <br/>**Note: When using Watson Studio, you already have a COS instance associated with the project you are running the notebook in.**


### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install plotly | tail -n 1
%pip install tqdm | tail -n 1
%pip install "nbformat>=4.2.0" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform `api_key` and instance `location`.

You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve platform API Key and instance location.

API Key can be generated in the following way:
```
ibmcloud login
ibmcloud iam api-key-create API_KEY_NAME
```

In result, get the value of `api_key` from the output.


Location of your watsonx.ai Runtime instance can be retrieved in the following way:
```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```

In result, get the value of `location` from the output.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.

You can also get service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, then copy the created key and paste it below.

**Action**: Enter your `url` and `api_key` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

In [3]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

You need to create a space that will be used for your work. If you do not have a space, you can use [Deployment Spaces Dashboard](https://dataplatform.cloud.ibm.com/ml-runtime/spaces?context=cpdaas) to create one.

- Click **New Deployment Space**
- Create an empty space
- Select Cloud Object Storage
- Select watsonx.ai Runtime instance and press **Create**
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: assign space ID below

In [4]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai Runtime, you need to set the **space** which you will be using.

In [5]:
client.set.default_space(space_id)

'SUCCESS'

<a id="2.-Timeseries-data-set"></a>
## 2. Timeseries data set

### Training data sets

Download training data from git repository and upload it to your space.  
This example uses the [China daily PM2.5](https://raw.githubusercontent.com/wjcougar/air_pollution/master/PM25.csv).

In [6]:
import os

import wget

filename = "PM25.csv"
base_url = "https://raw.githubusercontent.com/wjcougar/air_pollution/master/"

if not os.path.isfile(filename):
    wget.download(base_url + filename)

### Visualize the data

In [7]:
import pandas as pd
import plotly.express as px

df = pd.read_csv(filename)
fig = px.line(df, x="date", y=df.columns)
fig.show()

### Training data connection
The code in next cell defines connections to created assets.


In [8]:
from ibm_watsonx_ai.helpers import ContainerLocation, DataConnection

data_connection = DataConnection(
    location=ContainerLocation(path=filename),
)
data_connection.set_client(client)
data_connection.write(data=filename, remote_name=filename)

  Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-23.0.0-cp312-cp312-macosx_12_0_x86_64.whl (35.8 MB)


<a id="3.-Optimizer-definition"></a>
## 3. Optimizer definition

### Optimizer configuration

Provide the input information for AutoAI optimizer:
- `name` - experiment name
- `prediction_type` - type of the problem
- `prediction_columns` - target columns names
- `timestamp_column_name` — date&time column name/index
- `feature_columns` – names/indices of supporting feature columns
- `forecast_window` — future date/time range to be predicted
- `holdout_size` - number of holdout records
- `lookback_window` – past date/time range used for model training, -1 means auto-determined
- `backtest_num` – number of backtests
- `supporting_features_at_forecast` – whether leveraging future values of supporting features
- `pipeline_types` – specify an indiviual or a group of pipelins by type

In [9]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.utils.autoai.enums import ForecastingPipelineTypes

experiment = AutoAI(credentials, space_id=space_id)
forecast_window = 7
backtest_num = 4

pipeline_optimizer = experiment.optimizer(
    name="PM25 prediction",
    prediction_type=AutoAI.PredictionType.FORECASTING,
    prediction_columns=["pollution"],
    timestamp_column_name="date",
    feature_columns=["pollution", "dew", "temp", "press", "wnd_spd", "snow", "rain"],
    lookback_window=-1,
    backtest_num=backtest_num,
    holdout_size=14,
    forecast_window=forecast_window,
    supporting_features_at_forecast=True,
    pipeline_types=[ForecastingPipelineTypes.Bats]
    + ForecastingPipelineTypes.get_exogenous(),
)

Configuration parameters can be retrieved via `pipeline_optimizer.get_params()`.

<a id="4.-Experiment-run"></a>
## 4. Experiment run

Call the `fit()` method to trigger the AutoAI experiment. You can either use interactive mode (synchronous job) or background mode (asychronous job) by specifying `background_model=True`.

In [10]:
fit_details = pipeline_optimizer.fit(training_data_reference=[data_connection])

Training job 58fec39d-d4fe-46a7-8839-bc78988f3b91 completed: 100%|████████| [07:15<00:00,  4.36s/it]


You can use the `get_run_status()` method to monitor AutoAI jobs in background mode.

In [11]:
pipeline_optimizer.get_run_status()

'completed'

<a id="5.-Pipelines-comparison"></a>
## 5. Pipelines comparison

You can list trained pipelines and evaluation metrics information in
the form of a Pandas DataFrame by calling the `summary()` method. You can
use the DataFrame to compare all discovered pipelines and select the one
you like for further deployment.

In [12]:
summary = pipeline_optimizer.summary()
summary

,Enhancements,Estimator,Winner,validation_symmetric_mean_absolute_percentage_error,holdout_avg_r2,holdout_avg_mean_absolute_error,holdout_avg_root_mean_squared_error,holdout_avg_symmetric_mean_absolute_percentage_error,holdout_mean_absolute_error,holdout_root_mean_squared_error,holdout_symmetric_mean_absolute_percentage_error,holdout_r2,backtest_avg_r2,backtest_avg_mean_absolute_error,backtest_avg_root_mean_squared_error,backtest_avg_symmetric_mean_absolute_percentage_error,backtest_mean_absolute_error,backtest_root_mean_squared_error,backtest_symmetric_mean_absolute_percentage_error,backtest_r2
Pipeline Name,,,,,,,,,,,,,,,,,,,,
Pipeline_4,"HPO, FE, SUP",Ensembler,True,42.446582,0.570773,37.987089,61.398038,35.241435,37.987089,61.398038,35.241435,0.570773,0.649023,32.824713,48.311141,30.368102,32.824713,48.311141,30.368102,0.649023
Pipeline_1,"HPO, FE, SUP",RandomForest,True,46.988681,0.266752,58.515224,80.248455,58.627235,58.515224,80.248455,58.627235,0.266752,0.283743,46.987163,56.483505,51.204449,46.987163,56.483505,51.204449,0.283743
Pipeline_5,"HPO, FE, SUP",Ensembler,True,47.712462,0.280117,44.042905,79.513748,37.166297,44.042905,79.513748,37.166297,0.280117,0.249149,45.259106,66.380754,41.247001,45.259106,66.380754,41.247001,0.249149


Check pipeline details by calling `get_pipeline_details()`. By default details of best pipeline are returned.

In [13]:
best_pipeline_name = summary[summary.Winner].index.values[0]
print("Best pipeline is:", best_pipeline_name)

pipeline_details = pipeline_optimizer.get_pipeline_details()

Best pipeline is: Pipeline_4


### Holdout data visualization

In [14]:
visualization = pipeline_details["visualization"]["holdout"]
holdout_dates = visualization["time"]
holdout_predictions = visualization["predicted_values"][0]
holdout_observed_values = visualization["observed_values"][0]
holdout_df = pd.DataFrame(
    {
        "time": holdout_dates,
        "observed_values": holdout_observed_values,
        "predicted_values": holdout_predictions,
    }
)
fig = px.line(
    holdout_df,
    x="time",
    y=holdout_df.columns,
    hover_data={"time": "|%B %d, %Y"},
    title="Holdout data",
)
fig.update_xaxes(dtick="M1", tickformat="%b\n%Y")
fig.show()

### Backtest data visualization

In [15]:
from datetime import datetime, timedelta

import numpy as np

backtest_dfs = []
for i in range(backtest_num):
    visualization = pipeline_details["visualization"]["backtest"]["iterations"][i]
    backtest_dates = visualization["time"]
    backtest_predictions = visualization["predicted_values"][0]
    observed_values = visualization["observed_values"][0]
    backtest_dfs.append(
        pd.DataFrame(
            {
                "time": backtest_dates,
                "observed_values": observed_values,
                "predicted_values": backtest_predictions,
            }
        )
    )
backtest_df = pd.concat(backtest_dfs)
fig = px.line(
    backtest_df,
    x="time",
    y=backtest_df.columns,
    hover_data={"time": "|%B %d, %Y"},
    title="Backtest data",
)
fig.update_xaxes(dtick="M1", tickformat="%b\n%Y")
fig.show()

<a id="6.-Deploy-and-Score"></a>
## 6. Deploy and Score

In this section you will learn how to deploy and score pipeline model as online deployment using watsonx.ai Runtime instance.


### Online deployment creation

In [16]:
from ibm_watsonx_ai.deployment import WebService

service = WebService(credentials, source_space_id=space_id)

service.create(
    experiment_run_id=pipeline_optimizer.get_run_details()["metadata"]["id"],
    model=best_pipeline_name,
    deployment_name="PM2.5 Forecasting",
)

Preparing an AutoAI Deployment...
Published model uid: 0d00979a-7072-4739-ace7-8499c0bd3099
Deploying model 0d00979a-7072-4739-ace7-8499c0bd3099 using V4 client.


######################################################################################

Synchronous deployment creation for id: '0d00979a-7072-4739-ace7-8499c0bd3099' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='b79f7259-64d0-4560-81b4-6fc3cffd3fe2'
-----------------------------------------------------------------------------------------------




To show all available information about the deployment use the `.get_params()` method:

In [17]:
service.get_params()

### Scoring

You can use the `score` method to get predictions for defined forecasting window. You can either send payload records or use empty list.

In [18]:
predictions = service.score(payload=pd.DataFrame({"daily_cases": []}))
print("predictions for next 7 days:\n")
predictions

predictions for next 7 days:



{'predictions': [{'fields': ['prediction'],
   'values': [[[78.08768407034081]],
    [[78.81706833724878]],
    [[79.39375339176483]],
    [[81.76372568880456]],
    [[82.43436788307312]],
    [[87.12904924201023]],
    [[88.37465896994209]]]}]}

Or you could send payload with new obeservations:

```
filename = 'PM25_NewObservations.csv'

if not os.path.isfile(filename): wget.download(base_url + filename)
predictions = service.score(pd.read_csv(filename).drop("date", axis=1))
predictions
```

### Visualization of predictions

In [19]:
last_date = datetime.strptime(holdout_df.tail(1).time.values.tolist()[0], "%Y-%m-%d")
prediction_dates = [
    (last_date + timedelta(days=1 + i)).date() for i in range(forecast_window)
]
prediction_values = [pred[0][0] for pred in predictions["predictions"][0]["values"]]
pred_df = pd.DataFrame(
    {
        "time": holdout_dates + prediction_dates,
        "observed_values": holdout_observed_values
        + [np.nan for _ in range(forecast_window)],
        "predicted_values": holdout_predictions + prediction_values,
    }
)

fig = px.line(
    pred_df,
    x="time",
    y=pred_df.columns,
    hover_data={"time": "|%B %d, %Y"},
    title="Forecast data",
)
fig.update_xaxes(dtick="M1", tickformat="%b\n%Y")
fig.show()

<a id="7.-Cleanup"></a>
## 7. Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="8.-Summary-and-next-steps"></a>
## 8. Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI experiments. 

Check out our _[Online Documentation](https://www.ibm.com/cloud/watson-studio/autoai)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Jun Wang**, is a Software Architect and Data Scientist at IBM with a track record of developing enterprise-level applications that substantially increases clients' ability to turn data into actionable knowledge

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Maksym Sydorchuk**, Software Engineer at IBM watsonx.ai

Copyright © 2020-2026 IBM. This notebook and its source code are released under the terms of the MIT License.